In [4]:
import ROOT
import csv
from collections import Counter

# --- PARTE 1: LER DADOS DO NTUPLE (CLUSTERS E SHAPES) ---
f_ntuple = ROOT.TFile.Open("/workspace/work/Zee.NTUPLE.root", "READ")
tree_ntuple = f_ntuple.Get("physics")
num_entries_ntuple = tree_ntuple.GetEntries()

ntuple_data_por_evento = {}
event_clusters_count = Counter()

# Loop sobre os clusters/elétrons do Ntuple
for i in range(num_entries_ntuple):
    tree_ntuple.GetEntry(i)
    evt_num = tree_ntuple.EventNumber
    event_clusters_count[evt_num] += 1
    
    # Para o baseline, vamos guardar apenas o 1º cluster de cada evento como representante
    if evt_num not in ntuple_data_por_evento:
        ring0 = tree_ntuple.cl_rings_Rings[0] if tree_ntuple.cl_rings_Rings.size() > 0 else 0.0
        
        ntuple_data_por_evento[evt_num] = {
            "E_Reco_MeV": tree_ntuple.cl_e,
            "E_Truth_MeV": tree_ntuple.truth_cl_e,
            "eRatio": tree_ntuple.cl_eratio,
            "weta2": tree_ntuple.cl_weta2,
            "Reta": tree_ntuple.cl_reta,
            "Ring_0": ring0,
            "PID_Tight": bool(tree_ntuple.el_tight)
        }

# --- PARTE 2: LER DADOS DO ESD (CONTAGEM DE CÉLULAS) ---
f_esd = ROOT.TFile.Open("/workspace/work/Zee.ESD.root", "READ")
tree_esd = f_esd.Get("CollectionTree")
num_events = tree_esd.GetEntries()
print(f"Processando {num_events} eventos do ESD para extrair células...\n")

baseline_data = []

# Loop sobre os 10 eventos do ESD
for evt_idx in range(num_events):
    tree_esd.GetEntry(evt_idx)
    
    # O evento no ESD corresponde sequencialmente ao EventNumber gerado (geralmente 0 a 9)
    evt_num = evt_idx # Assumindo ordem sequencial, que é o padrão em seeds fixas
    
    # Extrai contagem de células
    n_cells_reco = tree_esd.CaloCellContainer_Cells_
    n_cells_truth = tree_esd.CaloCellContainer_TruthCells_
    n_clusters = event_clusters_count.get(evt_num, 0)
    
    # Busca dados do Ntuple para esse evento (ou zera se o evento não gerou cluster)
    ntuple_info = ntuple_data_por_evento.get(evt_num, {
        "E_Reco_MeV": 0.0, "E_Truth_MeV": 0.0, "eRatio": 0.0, 
        "weta2": 0.0, "Reta": 0.0, "Ring_0": 0.0, "PID_Tight": False
    })
    
    baseline_data.append({
        "Evento": evt_num,
        "Celulas_Reco": n_cells_reco,
        "Celulas_Truth": n_cells_truth,
        "Mult_Clusters": n_clusters,
        **ntuple_info # Expande o dicionário do Ntuple aqui
    })

# --- PARTE 3: IMPRIMIR E SALVAR O CSV COMPLETO ---
print(f"{'Evt':<4} | {'Cells Rec':<9} | {'Cells Tru':<9} | {'Clus':<4} | {'E Rec (MeV)':<11} | {'E Tru (MeV)':<11} | {'eRatio':<7} | {'weta2':<7} | {'Reta':<7} | {'Ring_0':<7} | {'Tight'}")
print("-" * 105)

with open('tabela_baseline.csv', mode='w', newline='') as csv_file:
    colunas = ["Evento", "Celulas_Reco", "Celulas_Truth", "Mult_Clusters", "E_Reco_MeV", "E_Truth_MeV", "eRatio", "weta2", "Reta", "Ring_0", "PID_Tight"]
    writer = csv.DictWriter(csv_file, fieldnames=colunas)
    writer.writeheader()
    
    for d in baseline_data:
        print(f"{d['Evento']:<4} | {d['Celulas_Reco']:<9} | {d['Celulas_Truth']:<9} | {d['Mult_Clusters']:<4} | {d['E_Reco_MeV']:<11.2f} | {d['E_Truth_MeV']:<11.2f} | {d['eRatio']:<7.4f} | {d['weta2']:<7.4f} | {d['Reta']:<7.4f} | {d['Ring_0']:<7.2f} | {d['PID_Tight']}")
        writer.writerow(d)

print("\n[OK] Extração completa e unificada! Tabela salva em 'tabela_baseline.csv'.")

Processando 10 eventos do ESD para extrair células...

Evt  | Cells Rec | Cells Tru | Clus | E Rec (MeV) | E Tru (MeV) | eRatio  | weta2   | Reta    | Ring_0  | Tight
---------------------------------------------------------------------------------------------------------
0    | 874895200 | 874895200 | 2    | 48857.73    | 50816.52    | 0.6019  | 0.0132  | 0.9584  | 112.12  | False
1    | 874895200 | 874895200 | 2    | 113351.20   | 0.00        | 0.6795  | 0.0135  | 0.9543  | 42.77   | True
2    | 874895200 | 874895200 | 2    | 54420.50    | 57340.58    | 0.6512  | 0.0114  | 0.9592  | 93.19   | True
3    | 874895200 | 874895200 | 1    | 169423.67   | 0.00        | 0.9555  | 0.0137  | 0.9052  | 0.00    | True
4    | 874895200 | 874895200 | 2    | 73549.63    | 77430.23    | 0.5743  | 0.0131  | 0.9630  | 113.98  | False
5    | 874895200 | 874895200 | 1    | 109551.63   | 0.00        | 0.7023  | 0.0131  | 0.9252  | 0.00    | True
6    | 874895200 | 874895200 | 2    | 39806.81    | 41880.3